In [8]:
import operator
from langgraph.graph import StateGraph, START, END
from typing import Annotated, List, TypedDict
from langgraph.types import Send

In [4]:
class OverallState(TypedDict): # this state will be passed as the base state for the graph
    cart_id: str # Ideally should be injected via the Context and Context will be getting this as an argument from the api 
    raw_items: List[dict]
    final_receipt: str 
    processed_items: Annotated[List[str], operator.add] # here the reducer does the concatenation of the list of strings
    total_discount: Annotated[float, operator.add] # here the reducer does the addition of the floats
    total_amount: Annotated[float, operator.add] # here the reducer does the addition of the floats


class CartItemWorkerState(TypedDict): # this state will be passed as a private state to the fanout worker nodes for the map step
    item_name: str 
    price: float 

In [5]:
def fetch_cart_items(state: OverallState) -> dict:
    items_from_db = [
        {"item_name": "Apple macbook pro", "price": 1000},
        {"item_name": "Apple macbook air", "price": 800},
        {"item_name": "Apple macbook mini", "price": 600},
        {"item_name": "Apple Iphone 15 pro", "price": 1000},
        {"item_name": "Apple Iphone 15 pro max", "price": 80}
    ]

    return {
        "raw_items": items_from_db
    }

In [6]:
def fanout_calculate_item_total(state: CartItemWorkerState) -> dict:
    name = state['item_name']
    price = state['price']

    final_price = (price * 0.9) * 1.05 # 10% discount and 5% tax

    summary = f"{name}: ${final_price} (discounted by 10% and taxed by 5%)"

    return {
        "processed_items": [summary],
        "total_discount": price * 0.1, # 10% discount,
        "total_amount": final_price
    }

In [7]:
def generate_receipt(state: OverallState) -> dict:
    lines = ["="*40, f"RECEIPT FOR CART #{state['cart_id']}", "="*40]

    for idx, line in enumerate(state["processed_items"], start=1):
        lines.append(f"{idx}. {line}")
    
    lines.append("="*40)

    lines.append(f"Total Discount: ${state['total_discount']}")
    lines.append(f"Total Amount: ${state['total_amount']}")
    return {
        "final_receipt": "\n".join(lines)
    }

In [9]:
def map_items_to_fanout_nodes(state: OverallState): # this is not a node, it is a function that will be used to map the items to the fanout nodes
    return [
        Send (  # send takes two inputs
            "fanout_calculate_item_total",  # 1 is the node which will act as a fanout node
            {"item_name": item["item_name"], "price": item["price"]} # 2 is the private state input to the fanout node
        )
        for item in state["raw_items"]
    ]


In [10]:
builder = StateGraph(OverallState)

builder.add_node("fetch_cart_items", fetch_cart_items)
builder.add_node("fanout_calculate_item_total", fanout_calculate_item_total)
builder.add_node("generate_receipt", generate_receipt) 

builder.add_edge(START, "fetch_cart_items")
builder.add_conditional_edges("fetch_cart_items", map_items_to_fanout_nodes) # this will trigger all the fanout_calculate_item_total nodes in parallel
builder.add_edge("fanout_calculate_item_total", "generate_receipt")
builder.add_edge("generate_receipt", END)

graph = builder.compile()

In [11]:
initial_state = {"cart_id": "1234567890", "raw_items": [], "final_receipt": "", "processed_items": [], "total_discount": 0, "total_amount": 0} 

In [12]:
result = graph.invoke(initial_state)
print(result)

{'cart_id': '1234567890', 'raw_items': [{'item_name': 'Apple macbook pro', 'price': 1000}, {'item_name': 'Apple macbook air', 'price': 800}, {'item_name': 'Apple macbook mini', 'price': 600}, {'item_name': 'Apple Iphone 15 pro', 'price': 1000}, {'item_name': 'Apple Iphone 15 pro max', 'price': 80}], 'final_receipt': '========================================\nRECEIPT FOR CART #1234567890\n========================================\n1. Apple macbook pro: $945.0 (discounted by 10% and taxed by 5%)\n2. Apple macbook air: $756.0 (discounted by 10% and taxed by 5%)\n3. Apple macbook mini: $567.0 (discounted by 10% and taxed by 5%)\n4. Apple Iphone 15 pro: $945.0 (discounted by 10% and taxed by 5%)\n5. Apple Iphone 15 pro max: $75.60000000000001 (discounted by 10% and taxed by 5%)\n========================================\nTotal Discount: $348.0\nTotal Amount: $3288.6', 'processed_items': ['Apple macbook pro: $945.0 (discounted by 10% and taxed by 5%)', 'Apple macbook air: $756.0 (discounted by